# 01 — Model Performance Evaluation (Global + Group Level)

Computes **13 performance metrics** for every run defined in `manifesto_config.pkl`
at two levels of granularity — in a single pass, into a **single output file**:

| Row type | `Group_Variable` | `Group_Value` | File |
|---|---|---|---|
| Overall model performance | `Global` | `Global` | `model_performance.csv` |
| Per-segment / per-group performance | e.g. `Segment` | e.g. `Platinum` | `model_performance.csv` |

**No inputs to change here.** All configuration is driven by `manifest.csv` via `00_input_manifesto.ipynb`.

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.xai_utils import calculate_regression_metrics

pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)

NOTEBOOK_DIR = os.path.abspath('')
OUTPUT_DIR   = os.path.join(NOTEBOOK_DIR, '..', 'Outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Libraries loaded')

## 1. Load Manifesto

In [ ]:
with open(os.path.join(OUTPUT_DIR, 'manifesto_config.pkl'), 'rb') as f:
    manifesto = pickle.load(f)

dataset_configs = manifesto['dataset_configs']
group_vars      = manifesto['group_vars']

print(f'Runs scheduled : {len(dataset_configs)}')
print(f'Group vars     : {group_vars}')

## 2. Run Metrics Pipeline — Global + Group in One Pass

In [ ]:
MIN_GROUP_SAMPLES = 5
all_results = []          # single list — global and group rows together

for run in dataset_configs:
    run_id     = run['run_id']
    model      = run['model']
    dataset    = run['dataset']
    pred_col   = run['pred_col']
    target_col = run['target_col']
    n_features = run['n_features']
    tolerance  = run['tolerance']
    run_gvars  = run['group_vars']

    print(f'\n[{run_id}]')
    df = pd.read_csv(run['path'])

    if pred_col not in df.columns or target_col not in df.columns:
        print(f'  WARNING: required column missing — skipping.')
        continue

    # Auto-create Decile if not present
    if 'Decile' not in df.columns:
        df['Decile'] = pd.qcut(
            df[target_col].rank(method='first'), 10,
            labels=[f'D{i}' for i in range(1, 11)]
        )

    meta = {
        'Dataset'     : dataset,
        'Model'       : model,
        'Model_Domain': run['model_domain'],
        'Model_Type'  : run['model_type'],
    }

    # ── Global row (Group_Variable = Group_Value = 'Global') ─────
    g_metrics = calculate_regression_metrics(
        df[target_col], df[pred_col],
        n_features=n_features, tolerance=tolerance
    )
    all_results.append({
        **meta,
        'Group_Variable': 'Global',
        'Group_Value'   : 'Global',
        'Samples'       : len(df),
        **g_metrics
    })
    print(f'  Global   — Samples={len(df):,}  MAE={g_metrics["MAE"]:.4f}  Accuracy={g_metrics["Accuracy %"]:.2f}%')

    # ── Group rows ────────────────────────────────────────────────
    for gvar in run_gvars:
        if gvar not in df.columns:
            print(f'  WARNING: group_var "{gvar}" not in file — skipping.')
            continue
        n_buckets = 0
        for gval, gdf in df.groupby(gvar):
            if len(gdf) < MIN_GROUP_SAMPLES:
                continue
            gm = calculate_regression_metrics(
                gdf[target_col], gdf[pred_col],
                n_features=n_features, tolerance=tolerance
            )
            all_results.append({
                **meta,
                'Group_Variable': gvar,
                'Group_Value'   : gval,
                'Samples'       : len(gdf),
                **gm
            })
            n_buckets += 1
        print(f'  Group [{gvar}] — {n_buckets} buckets')

print('\nPipeline complete.')

## 3. Combined Performance Table

In [ ]:
OUTPUT_COLS = [
    'Dataset', 'Model', 'Model_Domain', 'Model_Type',
    'Group_Variable', 'Group_Value', 'Samples',
    'MAE', 'RMSE', 'MAPE', 'Accuracy %',
    'Precision %', 'Recall %', 'F1 Score %',
    'R-Squared', 'Adjusted R-Squared',
    'High Confidence %', 'Medium Confidence %', 'Low Confidence %'
]

perf_df = pd.DataFrame(all_results)
perf_df = perf_df[[c for c in OUTPUT_COLS if c in perf_df.columns]]
perf_df = perf_df.sort_values(
    ['Dataset', 'Model', 'Group_Variable', 'Group_Value']
).reset_index(drop=True)

out_path = os.path.join(OUTPUT_DIR, 'model_performance.csv')
perf_df.to_csv(out_path, index=False)

n_global = (perf_df['Group_Variable'] == 'Global').sum()
n_group  = (perf_df['Group_Variable'] != 'Global').sum()
print(f'Saved: model_performance.csv')
print(f'  Total rows   : {len(perf_df)}')
print(f'  Global rows  : {n_global}')
print(f'  Group rows   : {n_group}')
display(perf_df)

## 4. Quick Views

In [ ]:
# Global rows only
print('── Global Performance ──────────────────────────────')
display(perf_df[perf_df['Group_Variable'] == 'Global'])

# Group rows only
print('── Group Performance ───────────────────────────────')
display(perf_df[perf_df['Group_Variable'] != 'Global'])

## 5. Visual Comparison — Accuracy % by Model and Dataset

In [ ]:
global_df = perf_df[perf_df['Group_Variable'] == 'Global'].copy()

sns.set_style('whitegrid')
unique_datasets = global_df['Dataset'].unique()
n_ds = len(unique_datasets)

fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 5), squeeze=False)
for i, ds in enumerate(unique_datasets):
    data = global_df[global_df['Dataset'] == ds]
    sns.barplot(x='Model', y='Accuracy %', data=data, ax=axes[0][i])
    axes[0][i].set_title(f'Accuracy % — {ds}')
    axes[0][i].set_ylim(0, 100)
    axes[0][i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'accuracy_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: accuracy_comparison.png')